# AI 5102 - Exercise 3: Semantic Search with Embeddings


### Before You Begin

Please make a copy of this Python notebook into your Google Drive or onto your own computer. If you edit it directly without making a copy, your changes will be lost.

**IMPORTANT:** To assist our grading efforts, we have code and markdown cells with special annotations like these:

    # === STUDENT INPUT CELL: Exercise X ===
    <!-- === STUDENT INPUT CELL: Exercise Y === -->

You'll see these in the coding cells.  They are also in the reflection questions.  You can reveal them by double clicking on the text in the reflection question sections.

To avoid grading issues, we ask you to not remove or alter these. So please take extra precaution when you select-all and paste. Finally, only make changes in the cells with annotations. If you create additional cells during development, consolidate the final solution code and make sure your answer is in the notebook cell we are expecting.

## Exercise Overview

### Table of Contents

| # | Section | Topics |
|---|---------|--------|
| — | **Introduction** | Setup, instructions, library installation |
| 1 | **Embeddings Refresher** | Hand-crafted fruit embeddings, Euclidean distance |
| 2 | **Word Embeddings with GloVe** | Real word vectors, cosine similarity, analogy arithmetic, clustering & visualization, limitations |
| 3 | **Text Embeddings with Qwen3** | Transformer-based embeddings, semantic similarity, context sensitivity, cross-lingual mapping, variable-length text |


This notebook introduces **embeddings** — a foundational technique in modern AI that represents words, sentences, and documents as lists of numbers where similar things get similar numbers.

**Instructions:** Run each "SETUP CELL" in order from top to bottom. Some cells (especially the model downloads) may take a minute or two.

In [ ]:
 # SETUP CELL 1
%pip install gensim sentence-transformers transformers umap-learn matplotlib numpy scikit-learn annoy -q

In [ ]:
# SETUP CELL 2
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

In [ ]:
# SETUP CELL 3
import gensim.downloader as gensim_api

print("Downloading GloVe vectors (this may take a minute on first run)...")
glove_word_vectors = gensim_api.load('glove-wiki-gigaword-100')
print(f"Loaded {len(glove_word_vectors):,} word vectors of dimension {glove_word_vectors.vector_size}")

## Section 1: Embeddings Refresher

How does Spotify know two songs "feel" similar? How does Google find pages matching your query even when they don't contain your exact words? These systems all rely on **embeddings** — representing things as lists of numbers where similar things get similar numbers.

### Hand-Crafted Embeddings

Let's start with an intuitive example. Imagine describing fruits using three numbers: **sweetness** (0–10), **size** (0–10), and **tartness** (0–10).

| Fruit | Sweetness | Size | Tartness |
|-------|-----------|------|----------|
| Apple | 7 | 5 | 4 |
| Banana | 9 | 6 | 0 |
| Lemon | 2 | 3 | 10 |
| Orange | 8 | 5 | 3 |
| Strawberry | 8 | 2 | 3 |
| Watermelon | 9 | 10 | 0 |

These three numbers form a **3-dimensional embedding** for each fruit. Some key terms:

- **Embedding**: A representation of an item as a list of numbers
- **Embedding dimension**: The number of values in the list (here, 3)
- **Vector**: Another name for a list of numbers — we use "embedding" and "vector" interchangeably
- **Distance**: A measure of how different two embeddings are — smaller distance means more similar

If our embeddings are good, **similar fruits should have similar vectors** (small distance between them). Let's check.

### Exercise 1.1: Computing Euclidean Distance
Since embeddings are vectors, Euclidean distance is one way to compute distance between them. In this exercise, you will define a function to compute the Euclidean distance between two vectors. You can do this manually or use numpy.

In [ ]:
# === STUDENT INPUT CELL: Exercise 1.1 ===

def euclidean_distance(a, b):
    ... # TODO: Implement this function

Now, let's apply this function to our fruit embeddings, and see which fruit is most similar to apple.

In [ ]:
fruits = {
    'apple':      np.array([7, 5, 4]),
    'banana':     np.array([9, 6, 0]),
    'lemon':      np.array([2, 3, 10]),
    'orange':     np.array([8, 5, 3]),
    'strawberry': np.array([8, 2, 3]),
    'watermelon': np.array([9, 10, 0]),
}

print("Distance from apple to every other fruit:")
print("-" * 40)
for name, vec in fruits.items():
    if name != 'apple':
        dist = euclidean_distance(fruits['apple'], vec)
        print(f"  apple -> {name:12s}  {dist:.2f}")

print()
closest = min(
    [(name, euclidean_distance(fruits['apple'], vec))
     for name, vec in fruits.items() if name != 'apple'],
    key=lambda x: x[1]
)
print(f"Closest to apple: {closest[0]} (distance: {closest[1]:.2f})")

### Exercise 1.2

Add "kiwi" and "pear" to the `fruits` dictionary with your own sweetness/size/tartness scores. For each new fruit, compute its distance to "apple". Which new fruit is most similar to "apple"?

In [ ]:
# === STUDENT INPUT CELL: Exercise 1.2 ===

### Exercise 1.3: Reflection questions

1. What is the smallest and largest distance you can get between any two embeddings using the Euclidean distance function you implemented?
2. Share how you chose the sweetness/size/tartness scores for the new fruits. Reflect on the process if you had to do this for, say, 100 different fruits.

<!-- === STUDENT INPUT CELL: Exercise 1.3 === -->
Answers to the reflection questions:

1.

2.

## Section 2: Word Embeddings with GloVe

### 2A: Looking at Real Word Vectors

The fruit embeddings above were **hand-crafted** — we chose the dimensions and values ourselves. But what if a machine could learn embeddings automatically from data?

That's exactly what **GloVe** (Global Vectors for Word Representation) does. It was trained on billions of words of text (Wikipedia + news articles) with a simple insight: **words that appear in similar contexts should get similar vectors.** For example, the blank in "The ___ chased the mouse" could be filled by *cat*, *dog*, or *kitten* — so these words should end up close together in embedding space.

GloVe learned 100-dimensional vectors for 400,000 words. Larger models exist (e.g., 300 dimensions trained on even more data), but 100 dimensions is enough to demonstrate the key concepts. Let's look at one.

In [ ]:
king_vec = glove_word_vectors['king']
print(f"Shape:           {king_vec.shape}")
print(f"First 10 values: {king_vec[:10].round(4)}")
print(f"Min value:       {king_vec.min():.4f}")
print(f"Max value:       {king_vec.max():.4f}")

A few points to note regarding embeddings, as illustrated by the `king_vec` example:
- The vector is 100-dimensional, as opposed to the 3-dimensional fruit vectors we made earlier
- The number of dimensions is fixed for all words.
- The values are real numbers, not integers.
- The vector is "dense", as opposed to "sparse", meaning it has mostly non-zero values.
- The dimensions don't have any particular meaning, and are not human-interpretable, unlike the fruit vectors we made earlier.

### 2B: Cosine Similarity

Another way to measure proximity between two vectors is to use a **similarity metric** as opposed to a distance metric, like the Euclidean distance we computed earlier. One popular choice is **cosine similarity**, which measures the *angle* between two vectors. Smaller the angle, more similar the vectors.

To compute it, we need the [**dot product**](https://en.wikipedia.org/wiki/Dot_product) — the sum of element-wise products of two vectors:

`A . B = A[0]*B[0] + A[1]*B[1] + ... + A[n]*B[n]`

The formula for cosine similarity is:

`cos_sim(A, B) = (A . B) / (||A|| x ||B||)`

where `||A||` is the length (norm) of vector A.

Here's how we can implement cosine similarity using numpy:

In [ ]:
def cosine_sim(a, b):
    """Compute cosine similarity between two vectors (0.0 if either is a zero vector)."""
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return np.dot(a, b) / denom

Cosine similarity, since it is the cosine of the angle between two vectors, ranges from **-1.0** to **1.0**:
- **1.0** means the vectors point in the same direction (most similar)
- **0.0** means they are unrelated
- **-1.0** means they point in opposite directions

### Exercise 2.1: Reflection questions

1. Why is a cosine similarity of 1.0 indicate "most similar"?
2. Why is a cosine similarity of 0.0 indicate "unrelated"?
3. Why is a cosine similarity of -1.0 indicate "most dissimilar"?
   
<!-- === STUDENT INPUT CELL: Exercise 2.1 === -->
Answers to the reflection questions:

1.
2.
3.

Let's test the cosine similarity function with embeddings for "king", "queen", and "bicycle". We should see that "king" is more similar to "queen" than to "bicycle".

In [ ]:
king_vec = glove_word_vectors['king']
queen_vec = glove_word_vectors['queen']
bicycle_vec = glove_word_vectors['bicycle']

print(f"cosine_sim(king, queen):   {cosine_sim(king_vec, queen_vec):.4f}")
print(f"cosine_sim(king, bicycle): {cosine_sim(king_vec, bicycle_vec):.4f}")

A natural question to ask is what words are most similar to "king"? We can use the `most_similar` method from `gensim` to find out. Gensim provides its own implementation of cosine similarity with the `similarity` method. In this example, we will use that.

In [ ]:
# Gensim provides built-in similarity functions
print("Top 10 words most similar to 'king':")
for word, score in glove_word_vectors.most_similar('king'):
    print(f"  {word:15s} {score:.4f}")

print(f"\nwv.similarity('king', 'queen') = {glove_word_vectors.similarity('king', 'queen'):.4f}")

### Exercise 2.2: Reflection question
Verify if similarity value returned by `similarity` method is the same as the cosine similarity value returned by our implementation.   

<!-- === STUDENT INPUT CELL: Exercise 2.2 === -->
Answer to the reflection question:
[Write your answer here]

### 2C: Analogy Arithmetic

One of the most surprising properties of word embeddings is that **vector arithmetic captures relationships between words.** The classic example:

> king - man + woman &approx; queen

The idea: the vector difference `king - man` captures the concept of "royalty." Adding that difference to `woman` lands near `queen`.

This allows us solve SAT-style analogy problems like:

king: man :: ______: woman

In [ ]:
print("king - man + woman = ?")
king_vec = glove_word_vectors['king']
man_vec = glove_word_vectors['man']
woman_vec = glove_word_vectors['woman']

# Compute the vector for the analogy: king - man + woman
target_vec = king_vec - man_vec + woman_vec

# Find the most similar word to this vector, ignoring the input words
results = glove_word_vectors.similar_by_vector(target_vec, topn=5)
for word, score in results:
    if word not in ['king', 'man', 'woman']: # exclude the input words
        print(f"  {word:15s} {score:.4f}")
        break

To help, `gensim` provides a `positive`/`negative` argument form in `most_similar` that allows you to compute the vector difference directly. For example, the following code solves the same problem as above:

In [ ]:
print("king - man + woman = ?")
results = glove_word_vectors.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)
for word, score in results:
    print(f"  {word:15s} {score:.4f}")

### Exercise 2.3: Solve the following analogy problems using embedding vector arithmetic.

france: paris :: germany: ______ <br>
big: bigger :: ______: smaller <br>
man: doctor :: woman: ______ <br>
son: daughter :: king: ______ <br>


In [ ]:
# === STUDENT INPUT CELL: Exercise 2.3 ===
# TODO: Write code to solve the analogy problems in Exercise 2.3.

<!-- === STUDENT INPUT CELL: Exercise 2.4 === -->
### Exercise 2.4: Reflection questions

1. Was the answer to the `man: doctor :: woman: ______` problem what you expected?<br>
   [Write your answer here]
   
2. Reflect on the broader implications of your answer to (1).<br>
   [Write your answer here]

### 2D: Clustering and Visualization

Vectors in 2 or 3 dimensions can be visualized by the human eye. But vectors in 100 dimensions can't be visualized directly. We need **dimensionality reduction** — a technique that projects high-dimensional data (100 numbers per word) down to 2 dimensions for visualization, while preserving which points are near each other. Future machine learning classes will cover dimensionality reduction in more detail, but for now, we'll use a common technique called **UMAP** (Uniform Manifold Approximation and Projection).

If words from the same category cluster together in the plot, that confirms the embeddings capture meaningful relationships. In the following example, we will consider words from three categories: technology, food, and sports.

In [ ]:
import umap

def visualize_word_embeddings(word_groups):
    words = []
    vectors = []
    labels = []
    colors_map = {'technology': '#2196F3', 'food': '#4CAF50', 'sports': '#F44336'}

    for group, word_list in word_groups.items():
        for word in word_list:
            words.append(word)
            vectors.append(glove_word_vectors[word])
            labels.append(group)

    vectors_array = np.array(vectors)
    np.random.seed(42)
    reducer = umap.UMAP(n_components=2, random_state=42)
    coords = reducer.fit_transform(vectors_array)

    plt.figure(figsize=(12, 8))
    for group, color in colors_map.items():
        mask = np.array([l == group for l in labels])
        plt.scatter(coords[mask, 0], coords[mask, 1], c=color, label=group, s=100, alpha=0.7)

    for i, word in enumerate(words):
        plt.annotate(word, (coords[i, 0], coords[i, 1]),
                    fontsize=8, ha='center', va='bottom', alpha=0.8)

    plt.legend(fontsize=12)
    plt.title("GloVe Word Embeddings Projected to 2D with UMAP", fontsize=14)
    plt.xlabel("UMAP Dimension 1")
    plt.ylabel("UMAP Dimension 2")
    plt.tight_layout()
    plt.show()

In [ ]:
word_groups = {
    'technology': ['computer', 'software', 'internet', 'algorithm', 'database',
                   'programming', 'keyboard', 'server', 'laptop', 'digital'],
    'food':       ['pizza', 'pasta', 'bread', 'cheese', 'chicken',
                   'rice', 'salad', 'soup', 'cake', 'chocolate'],
    'sports':     ['football', 'basketball', 'tennis', 'baseball', 'swimming',
                   'running', 'soccer', 'volleyball', 'golf', 'hockey'],
}

visualize_word_embeddings(word_groups)

<!-- === STUDENT INPUT CELL: Exercise 2.5 === -->
### Exercise 2.5: Reflection question
Why is "running" between "technology" and "sports" clusters in the plot?

[Write your answer here]

### Exercise 2.6: Developing intuition for word embeddings

Think of new words that might land between 1) "technology" and "sports" clusters and 2) "food" and "sports" clusters. Add them to the `word_groups` dictionary and visualize the new plot.

In [ ]:
# === STUDENT INPUT CELL: Exercise 2.6 === -->
# TODO: Modify `word_groups` to include new words

word_groups = {
    'technology': ['computer', 'software', 'internet', 'algorithm', 'database',
                   'programming', 'keyboard', 'server', 'laptop', 'digital'],
    'food':       ['pizza', 'pasta', 'bread', 'cheese', 'chicken',
                   'rice', 'salad', 'soup', 'cake', 'chocolate'],
    'sports':     ['football', 'basketball', 'tennis', 'baseball', 'swimming',
                   'running', 'soccer', 'volleyball', 'golf', 'hockey'],
}

visualize_word_embeddings(word_groups)

### 2E: Limitations of Word Embeddings

Word embeddings like GloVe are powerful, but they have important limitations:

1. **One vector per word** — The word "bank" always gets the same vector, whether it means a river bank or a financial bank. The embedding can't distinguish different meanings of the same word.

2. **No new words** — If a word wasn't in the training data, it has no vector. Try `glove_word_vectors['chatgpt']` and you'll get a `KeyError`.

3. **Can't handle phrases or sentences** — There's no principled way to combine word vectors into a sentence vector. Averaging "not" + "good" does not capture the meaning of "not good."

We'll address all three limitations in Section 3.

## Section 3: Text Embeddings with Qwen3

### 3A: From Words to Text

Modern **transformer** models solve all three limitations of word embeddings:
1. They produce **context-sensitive** vectors — "bank" gets different embeddings depending on the surrounding words
2. They handle **any text** — words, sentences, paragraphs
3. They work across **100+ languages**

We'll use [**Qwen3-Embedding-0.6B**](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B), a model with 600 million parameters that produces 1024-dimensional embeddings. (Larger models with billions of parameters exist and produce even better embeddings, but this one is small enough to run on most machines, including colab free tier instances.)

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading Qwen3-Embedding-0.6B (this may take a minute on first run)...")
text_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")
print(f"Embedding dimension: {text_model.get_sentence_embedding_dimension()}")

In [ ]:
def visualize_pairwise_text_similarity(sentences, labels, plot_title):
    embeddings = text_model.encode(sentences)
    print(f"Embedding shape: {embeddings.shape}")
    n = len(sentences)
    sim_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim_matrix[i][j] = cosine_sim(embeddings[i], embeddings[j])

    # Annotated heatmap
    _, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0, vmax=1)

    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{sim_matrix[i][j]:.3f}', ha='center', va='center', fontsize=11)

    ax.set_xticks(range(n))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticks(range(n))
    ax.set_yticklabels(labels)
    plt.colorbar(im, label='Cosine Similarity')
    plt.title(plot_title)
    plt.tight_layout()
    plt.show()

### 3B: Semantic Similarity

Let's encode some sentences and see how the model captures meaning.

In [ ]:
sentences = [
    "The cat sat on the mat",
    "A kitten is resting on the rug",
    "The stock market rallied today",
    "Financial markets showed strong gains",
    "We hiked through the mountain trail",
]
sentence_labels = ["cat/mat", "kitten/rug", "stock mkt", "financial", "hiking"]

visualize_pairwise_text_similarity(sentences, sentence_labels, plot_title="Semantic Similarity (Qwen3 Embeddings)")

Notice how "The cat sat on the mat" and "A kitten is resting on the rug" have high similarity despite sharing almost no words. The model understands that *cat &approx; kitten*, *sat &approx; resting*, and *mat &approx; rug*. This is **semantic** similarity — understanding meaning, not just matching keywords.

### 3C: Context Sensitivity

Recall how word embeddings from GloVe couldn't handle context sensitivity, i.e. the bank got the same vector regardless of it was a financial institution or a river bank. With sentence embeddings, we can handle this.

In [ ]:
bank_sentences = [
    "I deposited money at the bank",
    "The bank approved my loan application",
    "We had a picnic on the river bank",
    "Trees lined the bank of the stream",
]

bank_sentence_labels = ["deposit", "loan", "river", "stream"]

visualize_pairwise_text_similarity(bank_sentences, bank_sentence_labels, plot_title="Context Sensitivity: 'bank' in Different Meanings")

The model groups the sentences **by meaning**, not by the shared keyword "bank." The two financial sentences (deposit, loan) are similar to each other, and the two river sentences (river, stream) are similar to each other — even though all four sentences contain the word "bank."

This is exactly what word embeddings couldn't do. The same word gets a **different vector based on context**.

### 3D: Cross-Lingual Mapping

Qwen3 supports 100+ languages. A remarkable property: **the same meaning in different languages maps to similar embeddings.**

In [ ]:
cross_lingual_sentences = [
    "The weather is beautiful today",                                             # English
    "El clima es hermoso hoy",                                                    # Spanish
    "今天天气很好",                                                    # Chinese
    "Artificial intelligence is transforming healthcare",                          # English
    "La inteligencia artificial está transformando la atención médica",        # Spanish
    "人工智能正在改变医疗保健",                                    # Chinese
]

cross_lingual_labels = ["EN:weather", "ES:weather", "ZH:weather",
          "EN:AI",      "ES:AI",      "ZH:AI"]

visualize_pairwise_text_similarity(cross_lingual_sentences, cross_lingual_labels, plot_title="Cross-Lingual Similarity: Topic Grouping Across Languages")

### 3E: Not Just Sentences

Text embeddings work on any block of text — from a single word to a full paragraph. The output is always a fixed-size vector (1024 dimensions for Qwen3).

In [ ]:
short_text1 = "Neural networks can learn complex patterns."
short_text2 = "There were two big snowstorms in 2026."

long_text = (
    "Neural networks are computational models inspired by the human brain. "
    "They consist of layers of interconnected nodes that process information. "
    "These networks can learn complex patterns from data, making them useful "
    "for tasks like image recognition, natural language processing, and "
    "autonomous driving."
)

short1_emb = text_model.encode(short_text1)
short2_emb = text_model.encode(short_text2)
long_emb = text_model.encode(long_text)

print(f"Short text 1 embedding shape: {short1_emb.shape}")
print(f"Short text 2 embedding shape: {short2_emb.shape}")
print(f"Long text embedding shape:  {long_emb.shape}")
print(f"Cosine similarity between short text 1 and long text:          {cosine_sim(short1_emb, long_emb):.4f}")
print(f"Cosine similarity between short text 2 and long text:          {cosine_sim(short2_emb, long_emb):.4f}")
print()

if long_emb.shape == short1_emb.shape:
    print("Same fixed-size vector regardless of input length!")
else:
    print("Different fixed-size vector for different input lengths!")

if cosine_sim(short1_emb, long_emb) > cosine_sim(short2_emb, long_emb):
    print("Short text 1 is more similar to long text than short text 2.")
else:
    print("Short text 2 is more similar to long text than short text 1.")

## Semantic Search over a Document Collection

*(AI 5102 Exercise 3 — Semantic Search using Embeddings. Outcomes: CLO2, CLO3.)*

So far you've used embeddings to compare pairs of vectors: fruit vectors, GloVe word
vectors, and Qwen3 sentence vectors. In this section you'll put those pieces together to
build a small **semantic search engine**.

Given a **collection of documents** (a "corpus") and a **query**, semantic search:

1. Embeds every document once, ahead of time, into a matrix (the **index**).
2. Embeds the query with the *same* embedding model.
3. Ranks documents by **cosine similarity** between the query embedding and each
   document embedding — reusing the `cosine_sim` function from Section 2B.
4. Returns the top-`k` most similar documents.

You'll then compare this against a classic **keyword search** baseline, and evaluate
both with standard information-retrieval metrics.

This reuses the `text_model` (Qwen3 sentence embedder) and `cosine_sim` function you
already have from Section 3, so make sure you've run the setup cells and Section 3A
above before running the cells below.

### Step 1: A Small Document Collection

Below is a toy corpus of 16 short passages on a mix of topics (space, cooking, sports,
programming). Each passage has an integer id (its index in the list). We'll search over
this collection.


In [ ]:
# The document collection ("corpus") we will search over.
# Each entry is a short passage. The index in this list is the passage's id.
corpus = [
    "The James Webb Space Telescope captures infrared images of distant galaxies.",     # 0
    "NASA's Artemis program aims to return astronauts to the Moon by the late 2020s.",   # 1
    "Mars rovers use robotic arms to drill into rock and analyze soil samples.",         # 2
    "A solar eclipse occurs when the Moon passes between the Earth and the Sun.",        # 3
    "To make a good risotto, toast the rice in butter before adding warm stock.",        # 4
    "Kneading bread dough develops gluten, which gives the loaf its chewy structure.",    # 5
    "Searing a steak on high heat creates a flavorful crust through the Maillard reaction.", # 6
    "A classic vinaigrette is made from oil, vinegar, salt, and a touch of mustard.",     # 7
    "The striker scored a hat-trick, leading the team to a decisive victory.",            # 8
    "Marathon runners often practice negative splits, running the second half faster.",   # 9
    "The point guard's no-look pass set up an easy layup for the center.",                # 10
    "Swimmers use a flutter kick to maintain speed and balance during freestyle.",        # 11
    "Recursion is a technique where a function calls itself to solve smaller subproblems.", # 12
    "A hash map provides average O(1) lookup by mapping keys to array indices.",          # 13
    "Version control systems like Git let teams track changes and merge code safely.",    # 14
    "Binary search repeatedly halves a sorted array to find a target value quickly.",      # 15
]

print(f"Corpus has {len(corpus)} passages.")
for i, passage in enumerate(corpus):
    print(f"  [{i:2d}] {passage}")


### Step 2: Build the Index and Implement `semantic_search`

We embed every passage **once** with `text_model.encode(...)` to build the index — a
matrix where row `i` is the embedding of `corpus[i]`. Then `semantic_search(query, k)`
embeds the query and ranks the index rows by `cosine_sim`, reusing the function you
defined in Section 2B.

Notice that `semantic_search` takes `embedding_matrix` and `documents` as parameters
(rather than reading global variables) — this is what lets us unit-test the ranking
logic with small hand-made vectors, without loading any model.


In [ ]:
def build_index(documents, model):
    """Embed every document once and stack the embeddings into a matrix.

    Row i of the returned matrix is the embedding of documents[i].
    """
    return model.encode(documents)


def semantic_search(query, embedding_matrix, documents, model, k=5):
    """Return the top-k documents most similar to `query` by cosine similarity.

    Parameters
    ----------
    query : str
        The natural-language query.
    embedding_matrix : np.ndarray, shape (n_docs, dim)
        Precomputed document embeddings (the "index"), e.g. from build_index().
    documents : list[str]
        The original documents, same order as embedding_matrix's rows.
    model : SentenceTransformer
        Used only to embed the query, so it uses the same embedding space as the index.
    k : int
        Number of results to return.

    Returns
    -------
    list[tuple[int, float, str]]
        (doc_id, cosine_similarity, document_text) sorted by similarity, descending.
    """
    query_vec = model.encode(query)
    return semantic_search_from_vectors(query_vec, embedding_matrix, documents, k=k)


def semantic_search_from_vectors(query_vec, embedding_matrix, documents, k=5):
    """Core ranking logic, decoupled from any embedding model.

    Given a precomputed query vector and document matrix, rank documents by
    cosine_sim(query_vec, embedding_matrix[i]) and return the top-k.

    This is the function the unit tests exercise directly with toy vectors —
    it never touches SentenceTransformer, so it is fast, deterministic, and
    network-free.
    """
    scores = np.array([cosine_sim(query_vec, embedding_matrix[i]) for i in range(len(embedding_matrix))])
    top_k_indices = np.argsort(scores)[::-1][:k]
    return [(int(idx), float(scores[idx]), documents[idx]) for idx in top_k_indices]


# Build the index once (embedding all 16 passages).
corpus_embeddings = build_index(corpus, text_model)
print(f"Index shape: {corpus_embeddings.shape}  ({corpus_embeddings.shape[0]} docs x {corpus_embeddings.shape[1]} dims)")


Let's try `semantic_search` on a few natural-language queries.


In [ ]:
demo_queries = [
    "How do astronauts explore other planets?",
    "What makes bread chewy?",
    "efficient way to find an item in a sorted list",
]

for q in demo_queries:
    print(f'Query: "{q}"')
    for rank, (doc_id, score, text) in enumerate(semantic_search(q, corpus_embeddings, corpus, text_model, k=3), 1):
        print(f"  {rank}. [{score:.4f}] (id={doc_id}) {text}")
    print()


### Step 3: A Keyword Search Baseline

Semantic search understands *meaning*, but how much better is it than simple keyword
matching? Let's build a classic **TF-IDF** keyword ranker as a baseline: it scores each
document by term overlap with the query, weighting rare terms more heavily than common
ones.

`keyword_search(query, k)` below:
1. Tokenizes the corpus and query (lowercase, split on non-alphanumeric characters).
2. Computes TF-IDF vectors for every document.
3. Scores each document by the cosine similarity between the query's TF-IDF vector and
   the document's TF-IDF vector.
4. Returns the top-`k` documents.


In [ ]:
import re
from collections import Counter
import math


def tokenize(text):
    """Lowercase and split into alphanumeric tokens."""
    return re.findall(r"[a-z0-9]+", text.lower())


def build_tfidf_index(documents):
    """Build a TF-IDF matrix (dense) for a list of documents.

    Returns
    -------
    tfidf_matrix : np.ndarray, shape (n_docs, vocab_size)
    vocab : dict[str, int]
        Maps token -> column index.
    idf : np.ndarray, shape (vocab_size,)
        Inverse document frequency for each vocab term (reused for the query vector).
    """
    tokenized_docs = [tokenize(doc) for doc in documents]

    vocab = {}
    for tokens in tokenized_docs:
        for tok in tokens:
            if tok not in vocab:
                vocab[tok] = len(vocab)

    n_docs = len(documents)
    doc_freq = np.zeros(len(vocab))
    tf_matrix = np.zeros((n_docs, len(vocab)))

    for i, tokens in enumerate(tokenized_docs):
        counts = Counter(tokens)
        total = max(len(tokens), 1)
        for tok, cnt in counts.items():
            tf_matrix[i, vocab[tok]] = cnt / total
        for tok in counts:
            doc_freq[vocab[tok]] += 1

    idf = np.log((1 + n_docs) / (1 + doc_freq)) + 1  # smoothed idf, always positive
    tfidf_matrix = tf_matrix * idf
    return tfidf_matrix, vocab, idf


def vectorize_query_tfidf(query, vocab, idf):
    """Turn a query string into a TF-IDF vector in the same space as the doc matrix."""
    tokens = tokenize(query)
    vec = np.zeros(len(vocab))
    counts = Counter(tokens)
    total = max(len(tokens), 1)
    for tok, cnt in counts.items():
        if tok in vocab:
            vec[vocab[tok]] = (cnt / total) * idf[vocab[tok]]
    return vec


def keyword_search_from_vectors(query_vec, tfidf_matrix, documents, k=5):
    """Core keyword-ranking logic: cosine similarity over TF-IDF vectors.

    Decoupled from tokenization/vocab-building so it can be unit-tested directly
    with toy TF-IDF-like vectors.
    """
    scores = np.array([cosine_sim(query_vec, tfidf_matrix[i]) for i in range(len(tfidf_matrix))])
    top_k_indices = np.argsort(scores)[::-1][:k]
    return [(int(idx), float(scores[idx]), documents[idx]) for idx in top_k_indices]


def keyword_search(query, tfidf_matrix, vocab, idf, documents, k=5):
    """Rank `documents` against `query` using TF-IDF + cosine similarity."""
    query_vec = vectorize_query_tfidf(query, vocab, idf)
    return keyword_search_from_vectors(query_vec, tfidf_matrix, documents, k=k)


# Build the keyword index once.
tfidf_matrix, vocab, idf = build_tfidf_index(corpus)
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}  (vocab size = {len(vocab)})")


Now let's compare semantic search against the keyword baseline on the same queries.
Pay attention to cases where they disagree — this is exactly what happens with
synonyms/paraphrases (semantic wins) vs. rare shared terms (keyword can win).


In [ ]:
comparison_queries = [
    "How do astronauts explore other planets?",   # no shared words with passage 1/2, but semantically related
    "What makes bread chewy?",                     # shares "bread" with passage 5 (keyword should do fine here too)
    "efficient way to find an item in a sorted list",  # semantically = binary search, few shared words
]

for q in comparison_queries:
    print(f'Query: "{q}"')
    print("  Semantic search:")
    for rank, (doc_id, score, text) in enumerate(semantic_search(q, corpus_embeddings, corpus, text_model, k=3), 1):
        print(f"    {rank}. [{score:.4f}] (id={doc_id}) {text}")
    print("  Keyword search (TF-IDF):")
    for rank, (doc_id, score, text) in enumerate(keyword_search(q, tfidf_matrix, vocab, idf, corpus, k=3), 1):
        print(f"    {rank}. [{score:.4f}] (id={doc_id}) {text}")
    print()


### Exercise NEW.1: Reflection question

Look at the query `"efficient way to find an item in a sorted list"`. Compare the top
result from semantic search vs. keyword search.

1. Which method ranked passage 15 ("Binary search...") higher? Why do you think that happened?
2. Can you think of a query where **keyword search would win** — i.e. where sharing an
   exact rare word matters more than overall meaning?

<!-- === STUDENT INPUT CELL: Exercise NEW.1 === -->
Answer to the reflection question:
[Write your answer here]


### Step 4: Evaluating Retrieval Quality

To compare semantic vs. keyword search rigorously (rather than by eyeballing a few
examples), we need **retrieval metrics**. We'll use two standard ones:

- **Precision@k**: of the top-`k` results returned, what fraction are actually relevant?
  `precision_at_k = (# relevant in top-k) / k`
- **MRR (Mean Reciprocal Rank)**: for a single query, `1 / rank` of the *first* relevant
  result (0 if none appear in the results). Averaged over all queries, MRR rewards
  ranking relevant results near the top, not just anywhere in top-k.

We hand-label a small **evaluation set**: queries paired with the ids of passages we
know are relevant.


In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    """Fraction of the top-k retrieved ids that are in relevant_ids.

    Parameters
    ----------
    retrieved_ids : list[int]
        Document ids in ranked order (best first), length >= k.
    relevant_ids : set[int] | list[int]
        The set of ids that are considered relevant for this query.
    k : int
    """
    relevant_ids = set(relevant_ids)
    top_k = retrieved_ids[:k]
    if k == 0:
        return 0.0
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return hits / k


def mrr(retrieved_ids, relevant_ids):
    """Reciprocal rank of the first relevant id in retrieved_ids (0 if none found).

    Rank is 1-indexed: if the first relevant doc is at position 1 (top result),
    reciprocal rank is 1.0; at position 2, it's 0.5; etc.
    """
    relevant_ids = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def mean_mrr(list_of_retrieved_ids, list_of_relevant_ids):
    """Average MRR across multiple queries."""
    scores = [mrr(retrieved, relevant) for retrieved, relevant in zip(list_of_retrieved_ids, list_of_relevant_ids)]
    return sum(scores) / len(scores) if scores else 0.0


# A small hand-labeled evaluation set: query -> set of relevant passage ids.
eval_set = [
    {"query": "exploring space and other planets", "relevant_ids": {0, 1, 2, 3}},
    {"query": "how to cook meat and vegetables", "relevant_ids": {4, 5, 6, 7}},
    {"query": "athletes competing in a game", "relevant_ids": {8, 9, 10, 11}},
    {"query": "writing efficient computer programs", "relevant_ids": {12, 13, 14, 15}},
]

K = 4
semantic_precisions, keyword_precisions = [], []
semantic_retrieved_lists, keyword_retrieved_lists, relevant_lists = [], [], []

for item in eval_set:
    query, relevant_ids = item["query"], item["relevant_ids"]

    sem_results = semantic_search(query, corpus_embeddings, corpus, text_model, k=K)
    kw_results = keyword_search(query, tfidf_matrix, vocab, idf, corpus, k=K)

    sem_ids = [doc_id for doc_id, _, _ in sem_results]
    kw_ids = [doc_id for doc_id, _, _ in kw_results]

    semantic_retrieved_lists.append(sem_ids)
    keyword_retrieved_lists.append(kw_ids)
    relevant_lists.append(relevant_ids)

    semantic_precisions.append(precision_at_k(sem_ids, relevant_ids, K))
    keyword_precisions.append(precision_at_k(kw_ids, relevant_ids, K))

    print(f'Query: "{query}"')
    print(f"  Semantic top-{K} ids: {sem_ids}  -> precision@{K} = {precision_at_k(sem_ids, relevant_ids, K):.2f}")
    print(f"  Keyword  top-{K} ids: {kw_ids}  -> precision@{K} = {precision_at_k(kw_ids, relevant_ids, K):.2f}")
    print()

print("=" * 50)
print(f"Mean precision@{K}  — semantic: {np.mean(semantic_precisions):.3f}   keyword: {np.mean(keyword_precisions):.3f}")
print(f"MRR               — semantic: {mean_mrr(semantic_retrieved_lists, relevant_lists):.3f}   keyword: {mean_mrr(keyword_retrieved_lists, relevant_lists):.3f}")


### Step 5: The Deliverable — A Reusable Search Tool

Finally, let's wrap everything into a single, simple **search tool** function that
students (or graders) can call with any natural-language query. This is the deliverable for Exercise 3: given a query, it returns the top-`k` most relevant passages, using
semantic (embedding-based) search by default.


In [ ]:
def search_tool(query, k=5, method="semantic"):
    """Search the corpus for the top-k passages most relevant to `query`.

    Parameters
    ----------
    query : str
        A natural-language query.
    k : int
        Number of results to return.
    method : {"semantic", "keyword"}
        Which ranking method to use.

    Returns
    -------
    list[dict]
        Each dict has keys: "rank", "doc_id", "score", "text".
    """
    if method == "semantic":
        results = semantic_search(query, corpus_embeddings, corpus, text_model, k=k)
    elif method == "keyword":
        results = keyword_search(query, tfidf_matrix, vocab, idf, corpus, k=k)
    else:
        raise ValueError(f"Unknown method: {method!r}. Use 'semantic' or 'keyword'.")

    return [
        {"rank": rank, "doc_id": doc_id, "score": score, "text": text}
        for rank, (doc_id, score, text) in enumerate(results, 1)
    ]


# Try it out:
for result in search_tool("training for a long-distance race", k=3):
    print(f"{result['rank']}. [{result['score']:.4f}] (id={result['doc_id']}) {result['text']}")


### Exercise NEW.2: Try your own query

Call `search_tool` with a query of your own choosing (about any topic covered in the
corpus above), using both `method="semantic"` and `method="keyword"`. Compare the
results.


In [ ]:
# === STUDENT INPUT CELL: Exercise NEW.2 ===
# TODO: Call search_tool with your own query, using both "semantic" and "keyword" methods.


### Exercise Checklist

Use this checklist to track your progress. All exercises marked with `STUDENT INPUT CELL` require your response.

- [ ] **Exercise 1.1** — Implement the `euclidean_distance` function *(code)*
- [ ] **Exercise 1.2** — Add "kiwi" and "pear" to the fruits dictionary and compute distances *(code)*
- [ ] **Exercise 1.3** — Reflection: distance bounds and the challenge of hand-crafting embeddings *(written)*
- [ ] **Exercise 2.1** — Reflection: why cosine similarity values of 1.0, 0.0, and −1.0 mean what they do *(written)*
- [ ] **Exercise 2.2** — Verify that `gensim`'s `similarity` method matches your `cosine_sim` implementation *(written)*
- [ ] **Exercise 2.3** — Solve four analogy problems using vector arithmetic *(code)*
- [ ] **Exercise 2.4** — Reflection: bias in word embeddings (`man:doctor :: woman:?`) *(written)*
- [ ] **Exercise 2.5** — Reflection: why "running" appears between clusters *(written)*
- [ ] **Exercise 2.6** — Add new words to the embedding visualization and observe placement *(code)*

**Semantic Search over a Document Collection**

- [ ] **Exercise NEW.1** — Reflection: when does keyword search beat semantic search? *(written)*
- [ ] **Exercise NEW.2** — Call `search_tool` with your own query using both methods *(code)*